In [45]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPool2D,Conv2D,BatchNormalization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [46]:
IDG=ImageDataGenerator(rescale=1./255,rotation_range=20,zoom_range=0.2,horizontal_flip=True,vertical_flip=True,validation_split=0.2,fill_mode='nearest')



In [47]:
train=IDG.flow_from_directory("LIver Ultrasound Image (Fatty Liver)/.",target_size=(224,224),batch_size=16,class_mode='categorical',subset='training',shuffle=True)

Found 551 images belonging to 3 classes.


In [48]:
val=IDG.flow_from_directory("LIver Ultrasound Image (Fatty Liver)/.",target_size=(224,224),batch_size=16,class_mode='categorical',subset='validation',shuffle=False)

Found 136 images belonging to 3 classes.


In [49]:
print(f"train shape: {train.samples}")
print(f"val shape: {val.samples}")
print(f"classes shape: {train.class_indices}")

train shape: 551
val shape: 136
classes shape: {'Mild': 0, 'Normal': 1, 'Severe': 2}


In [39]:
print("train shape",train.image_shape)
print("val shape",val.image_shape)

train shape (224, 224, 3)
val shape (224, 224, 3)


In [40]:
model=Sequential()
model.add(Conv2D(32,(3,3),activation='relu',input_shape=(224,224,3)))
model.add(MaxPool2D(pool_size=(2,2)))
model.add(Conv2D(64,(3,3),activation='relu'))
model.add(Dropout(0.2))
model.add(Conv2D(128,(3,3),activation='relu'))
model.add(MaxPool2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dense(64,activation='relu'))
model.add(Dense(64,activation='relu'))
model.add(Dense(3,activation='softmax'))

C:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [41]:
compile=model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [42]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 109, 109, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 107, 107, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 53, 53, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 359552)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │    23,011,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 55)             │         3,575 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           168 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,108,383 (88.15 MB)

 Trainable params: 23,108,383 (88.15 MB)

 Non-trainable params: 0 (0.00 B)

In [52]:
imgs,lbls=[],[]
for i in range(len(val)):
    x,y=val[i]
    imgs.append(x)
    lbls.append(y)

val_images=np.concatenate(imgs)
val_labels=np.concatenate(lbls)

In [53]:
val_img,test_img=val_images[:68],val_images[68:]
val_lbl,test_lbl=val_labels[:68],val_labels[68:]

In [ ]:
history=model.fit(train,validation_data=(val_img,val_lbl),epochs=10,batch_size=16)